# Centralized Model Manager & Zoo Playground
This notebook serves as the single source of truth for downloading, converting, verifying, and deploying embedding models for the 3D Furniture Multimodal Search Engine.

### Production Constraints Enforced:
1. Commercial Compliance: Only Apache 2.0 or MIT licensed models are permitted.

2. Generic Decoupling: Files are auto-renamed to generic names (clip_visual.onnx, clip_text.onnx) so backend code requires zero modification when models swap.

3. Monorepo Structure: All assets are compiled directly into the root models/ directory shared across Python and C++.

# (Model Playground MVP)
Ten notebook służy do automatycznego pobierania, weryfikacji i przełączania modeli CLIP dla wielomodalnej wyszukiwarki mebli 3D.

Dzięki niemu możemy elastycznie testować różne architektury (mniejsze/szybsze lub większe/dokładniejsze), zachowując spójne nazewnictwo plików.

In [1]:
# 1. Install required framework dependencies
!pip install huggingface_hub onnx optimum[onnxruntime] tqdm --quiet

import os
import sys
import json
import shutil
import onnx
from huggingface_hub import hf_hub_download, snapshot_download

# Resolve standard Monorepo paths (Targeting clip_mvp_app/models)
PROJECT_ROOT = os.path.dirname(os.getcwd())
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
TOKENIZER_DIR = os.path.join(MODELS_DIR, "clip_tokenizer")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

print(f"✅ Environment initialized.")
print(f"📁 Target deployment directory: {MODELS_DIR}")


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Environment initialized.
📁 Target deployment directory: c:\Users\krzyc\Desktop\clip_mvp_app\models


c:\Users\krzyc\Desktop\clip_mvp_app\clip_fast_api\clip_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Lista dostępnych modeli i ich konfiguracji
W tej sekcji definiujemy mapę modeli dostępnych na Hugging Face, które pasują do naszego zastosowania. Możesz tu łatwo dopisywać nowe pozycje w przyszłości.

- Option 1 (Baseline Multilingual): Sprawdzony, stabilny model radzący sobie z językiem polskim.

- Option 2 (Pure English / Fast Test): Klasyczny model od OpenAI, idealny do szybkich testów wydajnościowych (mniejszy rozmiar).

## 🔍 Przewodnik Dewelopera: Jak rozbudowywać Model Zoo?

Ten notebook został zaprojektowany tak, aby umożliwić bezproblemową wymianę modeli w przyszłości. Jeśli chcesz przetestować nową architekturę (np. mniejszą/szybszą lub bardziej precyzyjną), musisz odpowiednio zmapować ją w słowniku `MODEL_ZOO`.

### 1. Jak szukać kompatybilnych modeli ONNX?
Najbezpieczniejszym i najszybszym źródłem gotowych modeli w formacie ONNX jest profil **`Xenova`** na Hugging Face (odpowiedzialny za oficjalny ekosystem *Transformers.js*).
1. Wejdź na [huggingface.co/Xenova](https://huggingface.co/Xenova).
2. Wyszukaj interesującą Cię architekturę (np. wpisując `clip` lub `siglip`).
3. Wejdź w dane repozytorium i skopiuj jego identyfikator (np. `Xenova/clip-vit-base-patch32`). To będzie Twoje `repo_id`.

### 2. Sprawdzanie struktury plików grafu
Przed dodaniem wpisu do kodu przejdź do zakładki **"Files and versions"** wybranego modelu:
* Wejdź do folderu `onnx/`.
* Upewnij się, że znajdują się tam pliki dla enkodera wizualnego oraz tekstowego.
* Podaj ścieżki do nich w słowniku (zazwyczaj jest to `"onnx/vision_model.onnx"` oraz `"onnx/text_model.onnx"`).

---

## ⚠️ Bardzo Ważne: Audyt Licencji Komercyjnej (Compliance Checklist)

Ponieważ nasza aplikacja jest rozwijana jako **produkt komercyjny**, każdy nowy model dodawany do tego projektu **MUSI** pozwalać na swobodne wykorzystanie biznesowe bez ryzyka prawnego. Przed pobraniem jakiegokolwiek modelu bezwzględnie sprawdź tag `License` na karcie modelu (Model Card).

### Ściągawka Licencyjna dla Zespołu:

| Status komercyjny | Typ Licencji | Czy możemy użyć w kodzie? | Opis |
| :--- | :--- | :--- | :--- |
| 🟢 **Zgoda (100% bezpieczne)** | **MIT / Apache 2.0** | **TAK** | Licencje wysoce liberalne. Możemy modyfikować model, zamykać nasz kod i sprzedawać aplikację klientom bez opłat licencyjnych. |
| 🟡 **Zgoda warunkowa** | **OpenRAIL / CC-BY-4.0** | **TAK (Po weryfikacji)** | Można użyć komercyjnie, ale wymaga to dopisania autora do dokumentacji aplikacji (`CC-BY`) lub przestrzegania zasad etycznego użytku (`OpenRAIL`). |
| 🔴 **BEZWZGLĘDNY ZAKAZ** | **CC-BY-NC (dowolna wersja)** | 🛑 **NIE!** | Końcówka **"-NC"** oznacza *Non-Commercial* (Użycie Niekomercyjne). Wdrożenie takiego modelu w komercyjnym oprogramowaniu grozi poważnymi konsekwencjami prawnymi i finansowymi. |

> 💡 **Złota zasada:** Nasz obecny baseline (`Xenova/clip-ViT-B-32-multilingual-v1`) bazuje na architekturach OpenAI oraz Meta i jest wydany na licencjach MIT oraz Apache 2.0, co czyni go w 100% bezpiecznym dla celów biznesowych.

In [2]:
MODEL_ZOO = {
    "siglip2_base_multilingual": {
        "repo_id": "google/siglip2-base-patch16-256",
        "method": "optimum_export",
        "task": "feature-extraction",  # <--- Jawna informacja dla eksportera
        "vector_dimensions": 768,
        "required_resolution": "256x256",
        "description": "Google SigLIP 2 Base (Apache 2.0) - High precision Polish/English alignment."
    },
    "multilingual_vit_b32_baseline": {
        "repo_id": "Xenova/clip-ViT-B-32-multilingual-v1",
        "method": "xenova_download",
        "task": "automatic",
        "vector_dimensions": 512,
        "required_resolution": "224x224",
        "description": "M-CLIP Base (MIT) - Legacy multi-lingual baseline."
    }
}



# "KLUCZ_DLA_CIEBIE": {
#     "repo_id": "UŻYTKOWNIK/NAZWA_REPO_Z_URL",
#     "vision_file": "SZYBKA_SCIEZKA_Z_TABU_FILES/vision.onnx",
#     "text_file": "SZYBKA_SCIEZKA_Z_TABU_FILES/text.onnx",
#     "description": "Twoja własna notatka, żebyś pamiętał co to za model"
# }

SELECTED_MODEL_KEY = "siglip2_base_multilingual"
active_config = MODEL_ZOO[SELECTED_MODEL_KEY]
print(f"🎯 Target Model Selected: {active_config['description']}")

🎯 Target Model Selected: Google SigLIP 2 Base (Apache 2.0) - High precision Polish/English alignment.


# ⚙️ Execution Pipeline (Download vs. Export)
- This block dynamically executes the correct acquisition strategy based on the model's design, abstracts away vendor folder structures, and extracts the raw assets.

- Ten proces pobiera wybrane pliki z Hugging Face, automatycznie zmienia ich nazwy na format generyczny oczekiwany przez konfigurację backendu (clip_visual_vitb32.onnx, clip_text_vitb32.onnx) oraz generuje aktualny plik model_manifest.json.

In [7]:
import subprocess
import sys
import os
import shutil

FINAL_UNIFIED_PATH = os.path.join(MODELS_DIR, "model.onnx")
FINAL_VISUAL_PATH = os.path.join(MODELS_DIR, "clip_visual.onnx")
FINAL_TEXT_PATH = os.path.join(MODELS_DIR, "clip_text.onnx")

# Clean up any old files from previous legacy tests
for path in [FINAL_UNIFIED_PATH, FINAL_VISUAL_PATH, FINAL_TEXT_PATH]:
    if os.path.exists(path):
        os.remove(path)

if active_config["method"] == "xenova_download":
    print("📥 Fetching pre-converted vendor ONNX graphs...")
    vis_temp = hf_hub_download(repo_id=active_config["repo_id"], filename="onnx/vision_model.onnx", local_dir=MODELS_DIR)
    txt_temp = hf_hub_download(repo_id=active_config["repo_id"], filename="onnx/text_model.onnx", local_dir=MODELS_DIR)
    shutil.move(os.path.join(MODELS_DIR, "onnx", "vision_model.onnx"), FINAL_VISUAL_PATH)
    shutil.move(os.path.join(MODELS_DIR, "onnx", "text_model.onnx"), FINAL_TEXT_PATH)
    shutil.rmtree(os.path.join(MODELS_DIR, "onnx"), ignore_errors=True)
    
    print("📥 Syncing tokenizer configurations...")
    snapshot_download(repo_id=active_config["repo_id"], allow_patterns=["*.json", "*.txt"], ignore_patterns=["onnx/*"], local_dir=TOKENIZER_DIR)

elif active_config["method"] == "optimum_export":
    print("⚡ Executing local PyTorch to ONNX compilation via Hugging Face Optimum...")
    TEMP_EXPORT_DIR = os.path.join(MODELS_DIR, "temp_export")
    
    venv_bin_dir = os.path.dirname(sys.executable)
    optimum_cli_exe = os.path.join(venv_bin_dir, "optimum-cli.exe")
    if not os.path.exists(optimum_cli_exe):
        optimum_cli_exe = os.path.join(venv_bin_dir, "optimum-cli")

    cmd = [
        optimum_cli_exe, "export", "onnx",
        "--model", active_config["repo_id"],
        "--task", active_config["task"],
        TEMP_EXPORT_DIR
    ]
    
    process = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if process.returncode != 0:
        print("\n🛑 --- TERMINAL ERROR LOG --- 🛑")
        print(process.stdout)
        print(process.stderr)
        print("---------------------------------")
        raise RuntimeError("❌ ONNX Compilation failed.")
        
    generated_files = os.listdir(TEMP_EXPORT_DIR)
    source_unified = os.path.join(TEMP_EXPORT_DIR, "model.onnx")
    
    if os.path.exists(source_unified):
        # Move the SINGLE model file directly to the models root
        shutil.move(source_unified, FINAL_UNIFIED_PATH)
        print("➡️ Unified architecture detected (SigLIP). Saved single model.onnx successfully.")
    else:
        raise FileNotFoundError(f"❌ Expected model.onnx was not found. Directory contains: {generated_files}")
    
    # Migrate tokenizer configurations
    for item in os.listdir(TEMP_EXPORT_DIR):
        item_path = os.path.join(TEMP_EXPORT_DIR, item)
        if not item.endswith(".onnx"):
            dest_path = os.path.join(TOKENIZER_DIR, item)
            if os.path.isdir(item_path):
                shutil.copytree(item_path, dest_path, dirs_exist_ok=True)
            else:
                shutil.copy2(item_path, dest_path)
                
    shutil.rmtree(TEMP_EXPORT_DIR, ignore_errors=True)

print("\n🎉 Pipeline complete. Maximum space optimization achieved!")

⚡ Executing local PyTorch to ONNX compilation via Hugging Face Optimum...
➡️ Unified architecture detected (SigLIP). Saved single model.onnx successfully.

🎉 Pipeline complete. Maximum space optimization achieved!


# 📝 Runtime Manifest Verification
Generates a frozen model_manifest.json descriptor file within the deployment folder so runtime client components can audit the active database parameters dynamically.

In [8]:
manifest = {
    "deployment_identity": SELECTED_MODEL_KEY,
    "original_hub_repo": active_config["repo_id"],
    "is_unified_model": (active_config["method"] == "optimum_export"),
    "expected_tensor_shapes": {
        "embedding_dimensions": active_config["vector_dimensions"],
        "preprocessing_image_resolution": active_config["required_resolution"]
    },
    "assets": {
        "model_file": "model.onnx" if active_config["method"] == "optimum_export" else None,
        "visual_graph": None if active_config["method"] == "optimum_export" else "clip_visual.onnx",
        "text_graph": None if active_config["method"] == "optimum_export" else "clip_text.onnx",
        "tokenizer_config": "clip_tokenizer/"
    }
}

manifest_path = os.path.join(MODELS_DIR, "model_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"💾 Production manifest optimized and saved to: {manifest_path}")

💾 Production manifest optimized and saved to: c:\Users\krzyc\Desktop\clip_mvp_app\models\model_manifest.json


# 🔍 Integrity Verification Check
Loads the freshly compiled binaries and evaluates the computational node graphs to ensure no data corruption occurred during download or disk writes.

In [6]:
try:
    print("Checking Visual Graph Integrity...")
    vis_graph = onnx.load(FINAL_VISUAL_PATH)
    onnx.checker.check_model(vis_graph)
    
    print("Checking Text Graph Integrity...")
    txt_graph = onnx.load(FINAL_TEXT_PATH)
    onnx.checker.check_model(txt_graph)
    
    print("\n🟢 INTEGRITY CHECK: PASSED. Models are healthy and ready for compilation/production use.")
except Exception as e:
    print(f"\n❌ INTEGRITY CHECK: FAILED. Graph structure is invalid: {e}")

Checking Visual Graph Integrity...
Checking Text Graph Integrity...

🟢 INTEGRITY CHECK: PASSED. Models are healthy and ready for compilation/production use.


# Testing sth else, just to see if my .py files are working correctly

In [3]:
# Diagnostic: inspect ONNX initializers and Add nodes
# This cell prints initializer shapes and searches for Add nodes related to embeddings.
from pathlib import Path
import os
import sys
try:
    import onnx
    from onnx import numpy_helper, shape_inference
except Exception as e:
    print("onnx is not installed in this environment:", e)

BASE_DIR = Path(os.getcwd()).parent
print("Current working directory:", BASE_DIR)

# =====================================================================
# 2. DIRECTORY STRUCTURE
# =====================================================================
# Zmień tę linię, dodając nazwę pliku na końcu:
model_path = Path(os.getenv("MODELS_DIR", BASE_DIR / "models")) / "model.onnx"

print("Model path:", model_path)
if not model_path.exists():
    print("Model file not found:", model_path)
else:
    m = onnx.load(str(model_path))
    try:
        m_inf = shape_inference.infer_shapes(m)
        graph = m_inf.graph
    except Exception as e:
        print("Shape inference failed:", e)
        graph = m.graph

    initializers = {}
    for init in graph.initializer:
        try:
            arr = numpy_helper.to_array(init)
            initializers[init.name] = arr
        except Exception:
            initializers[init.name] = None

    print("\nInitializers:")
    for name, arr in initializers.items():
        if arr is None:
            print(f" - {name}: <could not convert to array>")
        else:
            print(f" - {name}: {arr.shape}")

    # Collect inferred value info shapes
    value_shapes = {}
    for vi in list(graph.value_info) + list(graph.input) + list(graph.output):
        try:
            t = vi.type.tensor_type
            if t and t.HasField('shape'):
                dims = []
                for d in t.shape.dim:
                    if d.HasField('dim_value'):
                        dims.append(d.dim_value)
                    elif d.HasField('dim_param'):
                        dims.append(d.dim_param)
                    else:
                        dims.append('?')
                value_shapes[vi.name] = tuple(dims)
        except Exception:
            pass

    print("\nValue info shapes (inferred where possible):")
    for k, v in value_shapes.items():
        print(f" - {k}: {v}")

    print("\nSearching for Add nodes potentially related to embeddings:")
    for node in graph.node:
        if node.op_type == 'Add' and ('emb' in node.name.lower() or any('emb' in inp.lower() for inp in node.input)):
            print('\nNode:', node.name)
            print(' inputs:', node.input)
            print(' outputs:', node.output)
            for inp in node.input:
                if inp in initializers and initializers[inp] is not None:
                    print('  initializer', inp, initializers[inp].shape)
                elif inp in value_shapes:
                    print('  value_info', inp, value_shapes[inp])
                else:
                    print('  no static shape info for', inp)

    print('\nDone.')

Current working directory: c:\Users\krzyc\Desktop\clip_mvp_app
Model path: C:\Users\krzyc\Desktop\clip_mvp_app\models\model.onnx

Initializers:
 - text_model.embeddings.token_embedding.weight: (256000, 768)
 - text_model.embeddings.position_embedding.weight: (64, 768)
 - text_model.encoder.layers.0.layer_norm1.weight: (768,)
 - text_model.encoder.layers.0.layer_norm1.bias: (768,)
 - text_model.encoder.layers.0.self_attn.k_proj.bias: (768,)
 - text_model.encoder.layers.0.self_attn.v_proj.bias: (768,)
 - text_model.encoder.layers.0.self_attn.q_proj.bias: (768,)
 - text_model.encoder.layers.0.self_attn.out_proj.bias: (768,)
 - text_model.encoder.layers.0.layer_norm2.weight: (768,)
 - text_model.encoder.layers.0.layer_norm2.bias: (768,)
 - text_model.encoder.layers.0.mlp.fc1.bias: (3072,)
 - text_model.encoder.layers.0.mlp.fc2.bias: (768,)
 - text_model.encoder.layers.1.layer_norm1.weight: (768,)
 - text_model.encoder.layers.1.layer_norm1.bias: (768,)
 - text_model.encoder.layers.1.self_at

In [4]:
import sys
sys.path.insert(0, str(BASE_DIR))

from app.image_preprocessor import preprocess_image
from PIL import Image
import os

# Find a test image
images_dir = BASE_DIR / "images"
test_images = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))

if test_images:
    test_image_path = str(test_images[0])
    print(f"Testing with image: {test_image_path}")
    
    # Check raw image size
    img = Image.open(test_image_path).convert("RGB")
    print(f"\n1. Raw image size (PIL): {img.size}")  # (width, height)
    
    # Preprocess it
    print("\n2. Running preprocess_image()...")
    pixel_values = preprocess_image(test_image_path)
    
    print(f"\n3. Output shape: {pixel_values.shape}")
    print(f"   Expected: (1, 3, 256, 256)")
    
    # Calculate patches
    if pixel_values.shape[-1] == 256:
        patch_count = (256 // 16) ** 2
        print(f"\n✓ 256x256 image → {patch_count} patches (16x16)")
    elif pixel_values.shape[-1] == 224:
        patch_count = (224 // 16) ** 2
        print(f"\n✗ 224x224 image → {patch_count} patches (this is the problem!)")
    else:
        print(f"\n? Unexpected size: {pixel_values.shape[-1]}")
else:
    print("No test images found in images/ directory")

Testing with image: c:\Users\krzyc\Desktop\clip_mvp_app\images\IMG_20240911_113713.jpg

1. Raw image size (PIL): (4624, 3472)

2. Running preprocess_image()...
Initializing model processor...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The tokenizer you are loading from 'C:\Users\krzyc\Desktop\clip_mvp_app\models\clip_tokenizer' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Processor initialized.
[preprocess_image] Raw image size: (4624, 3472)
[preprocess_image] Preprocessed pixel_values shape: (1, 3, 256, 256)

3. Output shape: (1, 3, 256, 256)
   Expected: (1, 3, 256, 256)

✓ 256x256 image → 256 patches (16x16)


In [5]:
from transformers import AutoTokenizer
from app.clip_model import get_text_embedding
from app import config

print("\n=== Unified Text Inference Test ===")
print(f"Configured text max length: {config.TEXT_MAX_LENGTH}")

if test_images:
    tokenizer = AutoTokenizer.from_pretrained(str(config.TOKENIZER_DIR), local_files_only=True)
    sample_text = "A photo of a cat sitting on a chair."
    inputs = tokenizer(sample_text, return_tensors="np", padding="max_length", truncation=True, max_length=config.TEXT_MAX_LENGTH)
    print("Token shapes:", {k: v.shape for k, v in inputs.items()})
    try:
        text_emb = get_text_embedding(inputs)
        print("Text inference succeeded. Embedding shape:", text_emb.shape)
    except Exception as e:
        print("Text inference failed:")
        import traceback
        traceback.print_exc()
else:
    print("No test image available to confirm tokenizer.")



=== Unified Text Inference Test ===
Configured text max length: 64


The tokenizer you are loading from 'C:\Users\krzyc\Desktop\clip_mvp_app\models\clip_tokenizer' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Token shapes: {'input_ids': (1, 64)}
⚠️ DirectML niedostępne. CPU aktywowane dla: model.onnx
Text inference succeeded. Embedding shape: (1, 768)


In [7]:
print("\n=== Tokenizer Max Length Check ===")
try:
    tokenizer = AutoTokenizer.from_pretrained(str(config.TOKENIZER_DIR), local_files_only=True, fix_mistral_regex=False)
    print("Tokenizer class:", type(tokenizer))
    print("Tokenizer model_max_length:", getattr(tokenizer, "model_max_length", None))

    max_length = config.TEXT_MAX_LENGTH
    print("Using safe max_length from config:", max_length)

    sample = tokenizer(
        "A photo of a cat sitting on a chair.",
        return_tensors="np",
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )
    print("Tokenized shape:", {k:v.shape for k,v in sample.items()})
except Exception as e:
    print("Tokenizer load failed:")
    import traceback
    traceback.print_exc()



=== Tokenizer Max Length Check ===
Tokenizer class: <class 'transformers.models.gemma.tokenization_gemma_fast.GemmaTokenizerFast'>
Tokenizer model_max_length: 1000000000000000019884624838656
Using safe max_length from config: 64
Tokenized shape: {'input_ids': (1, 64)}


In [1]:
print("\n=== Unified Text Route Simulation ===")
from transformers import AutoTokenizer
from app.clip_model import get_text_embedding
from app import config

prompt = "fajny fotel u Babci taki czerwony ale stary"

tokenizer = AutoTokenizer.from_pretrained(
    str(config.TOKENIZER_DIR),
    local_files_only=True,
    fix_mistral_regex=False,
)

max_length = config.TEXT_MAX_LENGTH
configured_max = tokenizer.init_kwargs.get("model_max_length")
if isinstance(configured_max, int) and 1 <= configured_max <= 512:
    max_length = configured_max

print(f"Using max_length={max_length}")
inputs = tokenizer(
    prompt,
    return_tensors="np",
    padding="max_length",
    truncation=True,
    max_length=max_length,
)
print("Token shapes:", {k: v.shape for k, v in inputs.items()})
print("First tokens:", inputs["input_ids"][0][:min(20, inputs["input_ids"].shape[1])])

try:
    emb = get_text_embedding(inputs)
    print("Text embedding succeeded. Shape:", emb.shape)
except Exception as e:
    print("Text embedding failed:")
    import traceback
    traceback.print_exc()



=== Unified Text Route Simulation ===


c:\Users\krzyc\Desktop\clip_mvp_app\clip_fast_api\clip_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using max_length=64
Token shapes: {'input_ids': (1, 64)}
First tokens: [  1072  43734    517   3923    606  17147    667 103841 200178   8800
 194492      1      0      0      0      0      0      0      0      0]
⚠️ DirectML niedostępne. CPU aktywowane dla: model.onnx
Text embedding succeeded. Shape: (1, 768)


### ONNX diagnostic
This small cell prints the path checked and the shapes of initializers, any inferred value-info shapes, and details for Add nodes referencing embeddings. Run the next cell (Python) to collect diagnostic information from `models/model.onnx` or the file pointed to by the `MODEL_PATH` environment variable.